# Code based grading

Criteria for the code-generation prompt:

- **Format**: only Python, JSON or a regex, no explanation → code grader
- **Valid syntax**: the code parses → code grader
- **Task following**: it solves the task → model grader

Each validator tries to parse the output: 10 if it parses, 0 if not. The test case's `format`
field picks the validator. The final score is the average of the model and syntax scores.

The prompt also gets stricter: explicit "only code" rules plus a `` ```code `` prefill
(no need to know the language up front) and a `` ``` `` stop sequence.

Caveat: `re.compile` accepts almost any text, so plain prose passes the regex check.

## Setup

The client reads `ANTHROPIC_API_KEY` (and `ANTHROPIC_BASE_URL`, if set) from `.env` in the repo root.

In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Helpers

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    # The API rejects system=None, so optional params are only sent when set
    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return "".join(block.text for block in message.content if block.type == "text")

## Model grader (from the previous lesson)

In [3]:
import json
from statistics import mean


def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
"""

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

## Prompt v2: code only

In [4]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

## Syntax validators

In [5]:
import ast
import re


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

## Combine both scores

In [6]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "reasoning": reasoning,
    }

In [7]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

## Run it

In [8]:
with open("dataset.json") as f:
    dataset = json.load(f)

In [9]:
results = run_eval(dataset)

for result in results:
    test_case = result["test_case"]
    print(f"\n[{result['score']}] model={result['model_score']} syntax={result['syntax_score']} ({test_case['format']}) {test_case['task']}")
    print(f"    output: {result['output'].strip()[:150]!r}")
    print(f"    {result['reasoning']}")

Average score: 7.833333333333333

[8.0] model=6 syntax=10 (python) Write a Python function that extracts the AWS account ID from an ARN string
    output: 'import re\n\ndef extract_account_id(arn: str) -> str:\n    """\n    Extracts the AWS account ID from an ARN string.\n    \n    ARN format: arn:partition:ser'
    The solution works for standard ARN formats with account IDs present, but lacks robustness for edge cases and real-world AWS scenarios. The regex approach is simplistic and could match false positives if malformed input is provided. Better error handling and validation would improve reliability. However, for basic use cases with well-formed input, the implementation is functional and maintainable.

[8.0] model=6 syntax=10 (json) Create a JSON object that represents an AWS IAM policy allowing read-only access to S3 bucket objects
    output: '{\n  "Version": "2012-10-17",\n  "Statement": [\n    {\n      "Effect": "Allow",\n      "Action": [\n        "s3:GetObject",\n        